<a href="https://colab.research.google.com/github/ChanZH0525/Salary-Prediction-Prototype-Model/blob/main/Job_Salary_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd


df = pd.read_csv('postings.csv', on_bad_lines='warn', engine='python')
display(df.head())


/tmp/ipykernel_6990/3120891225.py:4: ParserWarning: Skipping line 25731: unexpected end of data

  df = pd.read_csv('postings.csv', on_bad_lines='warn', engine='python')


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25729 entries, 0 to 25728
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      25729 non-null  int64  
 1   company_name                25218 non-null  object 
 2   title                       25729 non-null  object 
 3   description                 25727 non-null  object 
 4   max_salary                  5753 non-null   float64
 5   pay_period                  7102 non-null   object 
 6   location                    25729 non-null  object 
 7   company_id                  25219 non-null  float64
 8   views                       25484 non-null  float64
 9   med_salary                  1349 non-null   float64
 10  min_salary                  5753 non-null   float64
 11  formatted_work_type         25729 non-null  object 
 12  applies                     5120 non-null   float64
 13  original_listed_time        257

In [ ]:
# Keep only requested columns and filter for yearly postings
relevant_cols = ['title', 'description', 'location', 'work_type', 'normalized_salary', 'pay_period', 'formatted_experience_level']
df_yearly = df[df['pay_period'] == 'YEARLY'][relevant_cols]
display(df_yearly.info())
display(df_yearly.head())

<class 'pandas.core.frame.DataFrame'>
Index: 4111 entries, 2 to 25719
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   title                       4111 non-null   object 
 1   description                 4111 non-null   object 
 2   location                    4111 non-null   object 
 3   work_type                   4111 non-null   object 
 4   normalized_salary           4111 non-null   float64
 5   pay_period                  4111 non-null   object 
 6   formatted_experience_level  2995 non-null   object 
dtypes: float64(1), object(6)
memory usage: 256.9+ KB


None

,title,description,location,work_type,normalized_salary,pay_period,formatted_experience_level
2,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",FULL_TIME,55000.0,YEARLY,NaN
3,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",FULL_TIME,157500.0,YEARLY,NaN
4,Service Technician,Looking for HVAC service tech with experience ...,"Burlington, IA",FULL_TIME,70000.0,YEARLY,NaN
6,Producer,Company DescriptionRaw Cereal is a creative de...,United States,CONTRACT,180000.0,YEARLY,NaN
7,Building Engineer,Summary: Due to the pending retirement of our ...,"San Francisco, CA",FULL_TIME,105000.0,YEARLY,NaN


In [ ]:
import pandas as pd
import numpy as np

# 1. Fill missing values
df_yearly['formatted_experience_level'] = df_yearly['formatted_experience_level'].fillna('Unspecified')

# 2. Create the "Neural Input" string using the specific columns
df_yearly['text_input'] = (
    "[TITLE] " + df_yearly['title'] +
    " [LOC] " + df_yearly['location'] +
    " [TYPE] " + df_yearly['work_type'] +
    " [EXP] " + df_yearly['formatted_experience_level'] +
    " [DESC] " + df_yearly['description'].str.slice(0, 500)
)

# 3. Log-Scale the target to prevent NaN/Gradient Explosion
# This converts $100,000 into ~11.5, which is much easier for the model to learn
df_yearly['log_salary'] = np.log1p(df_yearly['normalized_salary'])

In [ ]:
df_yearly.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4111 entries, 2 to 25719
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   title                       4111 non-null   object 
 1   description                 4111 non-null   object 
 2   location                    4111 non-null   object 
 3   work_type                   4111 non-null   object 
 4   normalized_salary           4111 non-null   float64
 5   pay_period                  4111 non-null   object 
 6   formatted_experience_level  4111 non-null   object 
 7   text_input                  4111 non-null   object 
 8   log_salary                  4111 non-null   float64
dtypes: float64(2), object(7)
memory usage: 321.2+ KB


In [ ]:
# Set pandas to display the full content of each cell
pd.set_option('display.max_colwidth', None)

# Display the first 5 entries of the concatenated text
print(df_yearly['text_input'].head())

2                                                                        [TITLE] Assitant Restaurant Manager [LOC] Cincinnati, OH [TYPE] FULL_TIME [EXP] Unspecified [DESC] The National Exemplar is accepting applications for an Assistant Restaurant Manager.\nWe offer highly competitive wages, healthcare, paid time off, complimentary dining privileges and bonus opportunities. \nWe are a serious, professional, long-standing neighborhood restaurant with over 41 years of service. If you are looking for a long-term fit with a best in class organization then you should apply now. \nPlease send a resumes to pardom@nationalexemplar.com. o
3    [TITLE] Senior Elder Law / Trusts and Estates Associate Attorney [LOC] New Hyde Park, NY [TYPE] FULL_TIME [EXP] Unspecified [DESC] Senior Associate Attorney - Elder Law / Trusts and Estates  Our legal team is committed to providing each client with quality counsel, innovative solutions, and personalized service. Founded in 2000, the firm offers the legal 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model_nm = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_nm)
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=1)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from sklearn.model_selection import train_test_split

# Fix: Use df_yearly instead of df to ensure no NaNs in log_salary
train_df, val_df = train_test_split(df_yearly, test_size=0.2, random_state=42)

def tokenize_fn(batch):
    return tokenizer(batch["text_input"], padding="max_length", truncation=True, max_length=512)

# Updated Trainer setup with lower learning rate to prevent NaN explosion
args = TrainingArguments(
    output_dir='salary_model',
    learning_rate=2e-6,        # Lowered from 2e-5 for stability
    per_device_train_batch_size=8,
    num_train_epochs=15,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False
)

In [ ]:
from datasets import Dataset

# Re-split to include the log_salary column
train_df, val_df = train_test_split(df_yearly, test_size=0.2, random_state=42)

# Convert Pandas -> Hugging Face Dataset using log_salary
train_dataset = Dataset.from_pandas(train_df[['text_input', 'log_salary']])
val_dataset = Dataset.from_pandas(val_df[['text_input', 'log_salary']])

# Rename 'log_salary' to 'labels'
train_dataset = train_dataset.rename_column("log_salary", "labels")
val_dataset = val_dataset.rename_column("log_salary", "labels")

# Apply tokenization
train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/3288 [00:00<?, ? examples/s]

Map:   0%|          | 0/823 [00:00<?, ? examples/s]

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.flatten()

    # Calculate R-Squared on the log scale
    r2 = r2_score(labels, predictions)

    # To get the real USD error, we must reverse the log using np.exp()
    real_prices = np.expm1(labels)
    predicted_prices = np.expm1(predictions)
    mae_usd = mean_absolute_error(real_prices, predicted_prices)

    return {"r2": r2, "avg_usd_error": mae_usd}

In [ ]:
from transformers import Trainer, DataCollatorWithPadding
import numpy as np

# Diagnostic: Final check for NaNs
print(f"NaNs in train labels: {np.isnan(train_dataset['labels']).any()}")
print(f"NaNs in val labels: {np.isnan(val_dataset['labels']).any()}")

# Force fp16 to False for stability with DeBERTa
args.fp16 = False

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Re-run training with the fresh model and low learning rate
trainer.train()

NaNs in train labels: False
NaNs in val labels: False


Epoch,Training Loss,Validation Loss,R2,Avg Usd Error
1,No log,37.786842,-23.141375,110214.593750
2,65.854867,16.397429,-9.476041,108704.679688
3,21.605830,5.190112,-2.315875,98469.656250
4,6.007021,1.601950,-0.023459,50227.089844
5,1.644643,1.654090,-0.056770,44725.238281
6,1.644643,1.557275,0.005084,42355.148438
7,1.410630,1.679422,-0.072955,45301.480469
8,1.421758,1.491181,0.047309,37333.218750
9,1.432498,1.494969,0.044890,36899.347656
10,1.293861,1.493361,0.045917,36791.855469


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=6165, training_loss=8.621437551679403, metrics={'train_runtime': 3225.7824, 'train_samples_per_second': 15.289, 'train_steps_per_second': 1.911, 'total_flos': 6533175589724160.0, 'train_loss': 8.621437551679403, 'epoch': 15.0})

In [ ]:
import torch

def predict_salary(title, location, description, experience="Unspecified"):
    # 1. Format the input exactly like the training data
    text = f"[TITLE] {title} [LOC] {location} [EXP] {experience} [DESC] {description[:500]}"

    # 2. Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda")

    # 3. Get Prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        log_pred = outputs.logits.item()

    # 4. Convert back from Log to USD
    salary_pred = np.exp(log_pred)
    return f"Predicted Salary: ${salary_pred:,.2f}"

# --- TEST IT OUT ---
print(predict_salary(
    title="Senior Product Marketing Manager",
    location="New Hyde Park, NY",
    description="A leading pharmaceutical company committed to developing and commercializing innovative and high-quality medicines that improve the lives of patients is now hiring for a Senior Product Marketing Manager. This role will be responsible for developing and executing marketing strategies for one or more pharmaceutical products. This individual will work closely with cross-functional teams including sales, medical affairs, market research, and commercial operations to develop and implement integrated marketing plans that drive product awareness, adoption, and revenue growth.Responsibilities: Develop and implement integrated marketing plans for our pharmaceutical productsAct as the Marketing point on patient support initiatives including, patient HUB, patient advocacy groups, and bridge programLeverage insights and analysis of customer needs and market environment to inform marketing strategies and develop tactical plansWork closely with cross-functional teams to develop and execute product-specific promotional campaigns across multiple channels including digital, print, and events.Manage product-specific budgets and track performance against marketing objectives and KPIs. Participate in the development of product launch plans and ensure effective execution of launch activities. Collaborate with Training to identify training needs and develop and deliver sales force training materials to address these needsï»¿Requirements: Minimum of 7 years of experience in pharmaceutical product marketing, with a demonstrated track record of success. Experience in immunology and preferably, in gastroenterology Strong knowledge of the pharmaceutical industry, including regulatory requirements, product development, and commercialization. Excellent communication and interpersonal skills, with the ability to collaborate effectively across cross-functional teams. Strong analytical skills and ability to analyze market research data to inform marketing strategies. Ability to manage budgets and track performance against marketing objectives. Ability to work independently and in a team environment.Willingness to travel up to 20% of is required."))

Predicted Salary: $128,089.52


In [ ]:
# Run these three to see how the model 'values' seniority
print(predict_salary(title="Junior Data Analyst", location="Remote", description="Entry level role..."))
print(predict_salary(title="Data Analyst", location="Remote", description="Mid level role..."))
print(predict_salary(title="Principal Data Analyst", location="Remote", description="Expert level role..."))

Predicted Salary: $22,064.27
Predicted Salary: $24,529.03
Predicted Salary: $36,763.98
